# Machine Learning – Practical 8: DBSCAN Clustering (Iris Dataset)

**Aim:** To perform density-based clustering with DBSCAN and detect noise points.

**Objective:**
- Cluster Iris data (Sepal Length and Width) with DBSCAN
- Identify clusters, core points and noise
- Evaluate with the Silhouette score

**Theory:** DBSCAN groups points that are densely packed. `eps` is the neighbourhood radius and `min_samples` is the minimum number of points needed to form a dense region. Points are core (dense), border (near a core) or noise (label −1). Unlike K-Means, it needs no preset number of clusters and finds arbitrary shapes.

## 1. Import Libraries

**Aim:** Import required libraries.

**Objective:** Load data, scaling, DBSCAN and metrics tools.

**Theory:** DBSCAN is available in `sklearn.cluster`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

print("All libraries imported successfully!")

## 2. Load Data & Standardise

**Aim:** Prepare two features for clustering.

**Objective:** Use Sepal Length and Sepal Width, then standardise.

**Theory:** Using two features allows easy 2D visualisation. Scaling is needed because DBSCAN depends on distances.

In [ ]:
iris = load_iris()

X = pd.DataFrame(
    iris.data[:, [0, 1]],
    columns=["Sepal Length", "Sepal Width"]
)

print("First five rows of the dataset:")
print(X.head())

print("\nDataset shape:", X.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature standardization completed!")

## 3. Apply DBSCAN

**Aim:** Cluster the data.

**Objective:** Run DBSCAN with eps = 0.4 and min_samples = 5.

**Theory:** A smaller `eps` creates tighter clusters and more noise; a larger `eps` merges clusters.

In [ ]:
dbscan = DBSCAN(
    eps=0.4,
    min_samples=5
)

cluster_labels = dbscan.fit_predict(X_scaled)

print("DBSCAN clustering completed!")

result = X.copy()
result["Cluster"] = cluster_labels

print("Cluster assignments:")
print(result.head(10))

## 4. Cluster Summary

**Aim:** Count clusters, noise and core points.

**Objective:** Report the number of clusters, points per cluster and noise points.

**Theory:** Label −1 marks noise (outliers). Core points have at least `min_samples` neighbours within `eps`.

In [ ]:
unique_labels = set(cluster_labels)

clusters = sorted(
    label for label in unique_labels
    if label != -1
)

noise_count = np.sum(cluster_labels == -1)
cluster_count = len(clusters)

print("\nDBSCAN Parameters")
print("eps =", dbscan.eps)
print("min_samples =", dbscan.min_samples)

print("\nNumber of clusters:", cluster_count)
print("Number of noise points:", noise_count)

print("Points in each cluster:")

for label in clusters:
    count = np.sum(cluster_labels == label)
    print(f"Cluster {label}: {count} points")

print(f"Noise (-1): {noise_count} points")

core_indices = dbscan.core_sample_indices_

print("Number of core points:", len(core_indices))

## 5. Visualise Clusters

**Aim:** Plot clusters, noise and core points.

**Objective:** Draw a scatter plot with noise as black crosses and core points circled in red.

**Theory:** The plot shows how DBSCAN separates dense regions from sparse outliers.

In [ ]:
plt.figure(figsize=(9, 6))

for label in sorted(unique_labels):
    mask = cluster_labels == label

    if label == -1:
        plt.scatter(
            X.iloc[mask, 0],
            X.iloc[mask, 1],
            c="black",
            marker="x",
            s=80,
            label="Noise"
        )
    else:
        plt.scatter(
            X.iloc[mask, 0],
            X.iloc[mask, 1],
            s=50,
            label=f"Cluster {label}"
        )

if len(core_indices) > 0:
    plt.scatter(
        X.iloc[core_indices, 0],
        X.iloc[core_indices, 1],
        s=110,
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        label="Core Points"
    )

plt.title("DBSCAN Clustering on Iris Dataset")
plt.xlabel("Sepal Length")
plt.ylabel("Sepal Width")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Silhouette Score

**Aim:** Evaluate cluster quality.

**Objective:** Compute the Silhouette score excluding noise points.

**Theory:** Noise points do not belong to any cluster, so they are excluded. At least two clusters are needed.

In [ ]:
non_noise_mask = cluster_labels != -1

non_noise_labels = cluster_labels[non_noise_mask]
non_noise_data = X_scaled[non_noise_mask]

if (
    len(set(non_noise_labels)) >= 2
    and len(non_noise_labels) > len(set(non_noise_labels))
):
    score = silhouette_score(
        non_noise_data,
        non_noise_labels
    )

    print(
        "\nSilhouette Score (excluding noise):",
        round(score, 4)
    )
else:
    print(
        "\nSilhouette Score not calculated: "
        "at least two non-noise clusters are required."
    )

## 7. Noise Points & Save Results

**Aim:** Inspect outliers and export results.

**Objective:** List the noise points and save the cluster assignments.

**Theory:** Noise points are the outliers DBSCAN detected; saving results allows further analysis.

In [ ]:
noise_data = result[result["Cluster"] == -1]

print("Noise Points:")
print(noise_data)

In [ ]:
result.to_csv(
    "DBSCAN_Clustering_Results.csv",
    index=False
)

print("Results saved to DBSCAN_Clustering_Results.csv")

## Conclusion

DBSCAN (eps = 0.4, min_samples = 5) found 2 clusters (40 and 80 points) and labelled 30 points as noise, with a Silhouette score of 0.5293. It detects outliers directly, but the result is sensitive to `eps` and `min_samples`.